#  Entrega de Proyecto Final: Identificación de Operadores Ineficaces
## Enlace para compartir y acceder:
### Link del proyecto en google drive: https://drive.google.com/drive/folders/14QT8fRFuvPCK4_odhFFdbsfRGmtf9dA6?usp=sharing
### Notebook disponible en este repositorio de GitHub
### Link de la presentación en PDF: https://drive.google.com/file/d/1RVk5SOzyE9NVb0eotJO5FFoOSo4si8sR/view?usp=sharing
### Link de Dasboard Tableu public: https://public.tableau.com/app/profile/wilma.cruz/viz/DashboarddeEficienciaOperativa-TelecomUS_/DashboarddeEficienciaOperativa-TelecomUS_?publish=yes

# Desarrollo del proyecto
## Proyecto Final: Telecomunicaciones: identificar operadores ineficaces
### Objetivo:
* Llevar a cabo el análisis exploratorio de datos
* Identificar operadores ineficaces
* Probar las hipótesis estadísticas

#### Nota:
Se considera que un operador es ineficaz:
* Si tiene una gran cantidad de llamadas entrantes perdidas (internas y externas) y un tiempo de espera prolongado para las llamadas entrantes.
* Como un operador debe realizar llamadas salientes, si tiene un número reducido de ellas.

## Paso 1: Carga de datos
El dataset comprimido `telecom_dataset_us.csv` contiene las siguientes columnas:

- `user_id`: ID de la cuenta de cliente
- `date`: fecha en la que se recuperaron las estadísticas
- `direction`: "dirección" de llamada (`out` para saliente, `in` para entrante)
- `internal`: si la llamada fue interna (entre los operadores de un cliente o clienta)
- `operator_id`: identificador del operador
- `is_missed_call`: si fue una llamada perdida
- `calls_count`: número de llamadas
- `call_duration`: duración de la llamada (sin incluir el tiempo de espera)
- `total_call_duration`: duración de la llamada (incluido el tiempo de espera)

 

El conjunto de datos `telecom_clients_us.csv` tiene las siguientes columnas:

- `user_id`: ID de usuario/a
- `tariff_plan`: tarifa actual de la clientela
- `date_start`: fecha de registro de la clientela

In [ ]:
# importo librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy import stats
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

# leo los datos de los dataset
calls = pd.read_csv('telecom_dataset_us.csv')
clients = pd.read_csv('telecom_clients_us.csv')

datasets = {
    'calls': calls,
    'clients': clients
}


## Paso 2: Análisis exploratorio de datos (EDA)

In [ ]:
# realizo inspección inicial de los dataframe
# defino una función que me ayude en la inspección de los 2 dataframe
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  DATASET: {name.upper()}")
    print(f"{'='*60}")
    print(f"  Forma:   {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"\n--- Primeras filas ---")
    display(df.head(3))
    print(f"\n--- Tipos de datos e info general ---")
    df.info()

# ── VALORES AUSENTES ────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  VALORES AUSENTES")
print("="*60)

for name, df in datasets.items():
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    result  = pd.DataFrame({'missing': missing, '%': pct})
    result  = result[result['missing'] > 0]
    if result.empty:
        print(f"\n ok {name}: sin valores ausentes")
    else:
        print(f"\n Alerta  {name}:")
        print(result)

# ── DUPLICADOS ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  DUPLICADOS")
print("="*60)

for name, df in datasets.items():
    # Duplicados totales (filas idénticas)
    dup_full = df.duplicated().sum()
    print(f"\n{name}:")
    print(f"  Filas completamente dupicadas: {dup_full}")

### En la exploración de archivos encontramos:
1. Dimensiones del Conjunto de Datos
El conjunto de datos está compuesto por dos fuentes principales:
  * Dataset de Llamadas (calls): Registra un volumen total de 53,902 transacciones telefónicas.
  * Dataset de Clientes (clients): Contiene la información estructural de 732 clientes únicos con su respectivo plan tarifario (ej. Plan A) y fecha de alta.

**Fuerza Operativa:** Se identifican 1,092 operadores únicos dentro del histórico de llamadas.

2. Calidad de los Datos y Hallazgos Críticos

El análisis de consistencia e integridad de los datos revela dos alertas prioritarias que deben resolverse en la fase de limpieza antes de consolidar las métricas de rendimiento:

* Alerta A: Operadores no asignados (operator_id nulos)
  * Métrica: La columna operator_id presenta 8,172 valores ausentes, lo que equivale al 15.16% del total del dataset.
  * Impacto Analítico: Al no tener un identificador, estas 8,172 llamadas no pueden ser asignadas a ningún operador para medir su eficiencia.
  * Explicación del Negocio: Dado que el dataset registra llamadas perdidas (is_missed_call), una parte masiva de estos nulos sugiere que corresponde a llamadas entrantes que colgaron en la cola de espera antes de que un operador físico las tomara.

* Alerta B: Duplicidad de Registros

  * Métrica: Se detectaron 4,900 filas completamente duplicadas en el dataset de llamadas. El dataset de clientes está 100% limpio (0 duplicados).
  * Impacto Analítico: Mantener estos duplicados inflaría artificialmente el conteo total de llamadas, alterando las tasas de pérdida y los promedios de tiempo de espera. Deben ser eliminados (.drop_duplicates()) antes de cualquier cálculo.

* Alerta C: Columna internal

* Presenta solo 117 valores nulos (0.22%). Es un volumen marginal que no pone en riesgo el análisis y puede ser imputado como False (llamadas externas) de forma segura.

3. Conclusiones y Próximos Pasos Estratégicos
   * Ventana de Tiempo Estable: El rango de fechas de agosto a noviembre proporciona un histórico maduro y continuo para evaluar comportamientos estables del equipo, sin sesgos por estacionalidades extremas (como fin de año).
   * Estrategia de Limpieza Obligatoria:
     * Eliminar los 4,900 duplicados para trabajar sobre la realidad exacta de la operación.
     * Segmentar los operator_id nulos: Antes de borrarlos, se debe validar cuántos de esos 8,172 registros corresponden a llamadas perdidas (is_missed_call == True). Si son llamadas perdidas en cola, sirven para medir el abandono general de la empresa, pero se deben excluir al evaluar el desempeño individual de los operadores.



## Paso 3: Limpieza y Pre procesamiento

In [ ]:
# convierto tipos de datos a datetime
# Columnas que deben ser datetime
date_cols = {
    'calls': ['date'],
    'clients':['date_start'],
}

for name, cols in date_cols.items():
    df = datasets[name]
    for col in cols:
        df[col] = pd.to_datetime(df[col])
    print(f" OK {name}: columnas {cols} convertidas a datetime")

# verificación post-conversión
for name, cols in date_cols.items():
    print(f"\n{name} — tipos actualizados:")
    print(datasets[name][cols].dtypes)

In [ ]:
# ─── 1. ESTADÍSTICAS ORIGINALES ───────────────────────────────
print("=" * 55)
print("DATASET ORIGINAL")
print("=" * 55)
print(f"Registros totales:        {len(calls):,}")
print(f"Clientes únicos:          {clients['user_id'].nunique():,}")
print(f"Operadores únicos:        {calls['operator_id'].nunique():,}")
print(f"\nRango de fechas:")
print(f"  Desde: {calls['date'].min()}")
print(f"  Hasta: {calls['date'].max()}")

print(f"\nValores nulos por columna:")
print(calls.isnull().sum().to_string())

print(f"\nDuplicados totales:       {calls.duplicated().sum():,}")
print(f"operator_id nulos:        {calls['operator_id'].isnull().sum():,}")



In [ ]:
# ─── 2. LIMPIEZA: ELIMINAR DUPLICADOS ─────────────────────────
print("\n" + "=" * 55)
print("PASO 1 — ELIMINAR DUPLICADOS")
print("=" * 55)

before = len(calls)
calls = calls.drop_duplicates()
after = len(calls)

print(f"Registros antes:          {before:,}")
print(f"Duplicados eliminados:    {before - after:,}")
print(f"Registros después:        {after:,}")



In [ ]:
# ─── 3. LIMPIEZA: ELIMINAR FILAS SIN OPERADOR ─────────────────
print("\n" + "=" * 55)
print("PASO 2 — ELIMINAR REGISTROS SIN OPERADOR ASIGNADO")
print("=" * 55)

before = len(calls)
calls_clean = calls[calls['operator_id'].notna()].copy()
after = len(calls_clean)

print(f"Registros antes:          {before:,}")
print(f"Sin operator_id (nulos):  {before - after:,}")
print(f"Registros válidos:        {after:,}")

# Convertir operator_id a entero y luego string para consistencia
calls_clean['operator_id'] = calls_clean['operator_id'].astype(int).astype(str)



In [ ]:
# ─── 4. OPERADORES ACTIVOS (≥ 10 LLAMADAS ENTRANTES) ──────────
print("\n" + "=" * 55)
print("PASO 3 — FILTRAR OPERADORES ACTIVOS")
print("=" * 55)

# Contar llamadas entrantes por operador
incoming_per_op = (
    calls_clean[calls_clean['direction'] == 'in']
    .groupby('operator_id')['calls_count']
    .sum()
    .rename('total_incoming')
)

total_operators = calls_clean['operator_id'].nunique()
active_operators = incoming_per_op[incoming_per_op >= 10]
excluded_operators = total_operators - len(active_operators)

print(f"Operadores totales:       {total_operators:,}")
print(f"  Con ≥ 10 llamadas ent.: {len(active_operators):,}  ← activos")
print(f"  Con < 10 llamadas ent.: {excluded_operators:,}  ← excluidos")

# ─── 5. RESUMEN FINAL ─────────────────────────────────────────
print("\n" + "=" * 55)
print("RESUMEN DEL PIPELINE DE LIMPIEZA")
print("=" * 55)
print(f"{'Etapa':<40} {'Registros':>10}")
print("-" * 55)
print(f"{'1. Dataset original':<40} {'53,902':>10}")
print(f"{'2. Tras eliminar duplicados':<40} {len(calls):>10,}")
print(f"{'3. Tras eliminar sin operator_id':<40} {len(calls_clean):>10,}")
print("-" * 55)
print(f"{'Clientes en telecom_clients_us.csv':<40} {clients['user_id'].nunique():>10,}")
print(f"{'Operadores únicos (dataset limpio)':<40} {calls_clean['operator_id'].nunique():>10,}")
print(f"{'Operadores activos (≥10 llamadas ent.)':<40} {len(active_operators):>10,}")

In [ ]:
# ─── 5. RESUMEN FINAL ─────────────────────────────────────────
print("\n" + "=" * 55)
print("RESUMEN DEL PIPELINE DE LIMPIEZA")
print("=" * 55)
print(f"{'Etapa':<40} {'Registros':>10}")
print("-" * 55)
print(f"{'1. Dataset original':<40} {'53,902':>10}")
print(f"{'2. Tras eliminar duplicados':<40} {len(calls):>10,}")
print(f"{'3. Tras eliminar sin operator_id':<40} {len(calls_clean):>10,}")
print("-" * 55)
print(f"{'Clientes en telecom_clients_us.csv':<40} {clients['user_id'].nunique():>10,}")
print(f"{'Operadores únicos (dataset limpio)':<40} {calls_clean['operator_id'].nunique():>10,}")
print(f"{'Operadores activos (≥10 llamadas ent.)':<40} {len(active_operators):>10,}")

## Paso 4: Análisis operadores ineficaces

In [ ]:
# obtengo el tiempo de espera
# Resto la duración neta de la duración total para obtener el tiempo de espera

calls_clean['wait_time'] = (calls_clean['total_call_duration'] - calls_clean['call_duration']).clip(lower=0)

# ── LLAMADAS PERDIDAS SEPARADAS POR TIPO ──────────────────────

# Perdidas EXTERNAS entrantes (internal=False)
missed_external = (
    calls_clean[
        (calls_clean['direction'] == 'in') &
        (calls_clean['is_missed_call'] == True) &
        (calls_clean['internal'] == False)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
    .rename('missed_external')
)

# Perdidas INTERNAS entrantes (internal=True)

# Corregimos columna internal
calls_clean['internal'] = (
    calls_clean['internal']
    .astype(str)
    .map({'True': True, 'False': False, 'true': True, 'false': False})
    .fillna(False)
)
missed_internal = (
    calls_clean[
        (calls_clean['direction'] == 'in') &
        (calls_clean['is_missed_call'] == True) &
        (calls_clean['internal'] == True)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
    .rename('missed_internal')
)

# Total entrantes externas
total_external_in = (
    calls_clean[
        (calls_clean['direction'] == 'in') &
        (calls_clean['internal'] == False)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
    .rename('total_external_in')
)

# Total entrantes internas
total_internal_in = (
    calls_clean[
        (calls_clean['direction'] == 'in') &
        (calls_clean['internal'] == True)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
    .rename('total_internal_in')
)

# Tiempo de espera (solo llamadas externas respondidas — más representativo)
avg_wait = (
    calls_clean[
        (calls_clean['direction'] == 'in') &
        (calls_clean['is_missed_call'] == False) &
        (calls_clean['internal'] == False)
    ]
    .groupby('operator_id')['wait_time']
    .mean()
    .rename('avg_wait_time')
)

# Llamadas salientes externas
outgoing_external = (
    calls_clean[
        (calls_clean['direction'] == 'out') &
        (calls_clean['internal'] == False)
    ]
    .groupby('operator_id')['calls_count']
    .sum()
    .rename('outgoing_calls')
)

operators_with_outgoing_role = calls_clean[
    (calls_clean['direction'] == 'out') & (calls_clean['internal'] == False)
]['operator_id'].unique()

# ── CONSOLIDAR MÉTRICAS ───────────────────────────────────────
metrics = (
    pd.DataFrame(index=calls_clean['operator_id'].unique())
    .join(missed_external)
    .join(missed_internal)
    .join(total_external_in)
    .join(total_internal_in)
    .join(avg_wait)
    .join(outgoing_external)
    .fillna(0)
)
metrics.index.name = 'operator_id'

# Tasas separadas
metrics['missed_rate_external'] = np.where(
    metrics['total_external_in'] > 0,
    metrics['missed_external'] / metrics['total_external_in'], 0
)

metrics['missed_rate_internal'] = np.where(
    metrics['total_internal_in'] > 0,
    metrics['missed_internal'] / metrics['total_internal_in'], 0
)

# Tasa combinada (ponderada por volumen, fiel a la definición del negocio)
metrics['total_incoming'] = metrics['total_external_in'] + metrics['total_internal_in']
metrics['missed_total']   = metrics['missed_external']   + metrics['missed_internal']
metrics['missed_rate_combined'] = np.where(
    metrics['total_incoming'] > 0,
    metrics['missed_total'] / metrics['total_incoming'], 0
)

metrics['has_outgoing_role'] = metrics.index.isin(operators_with_outgoing_role)

# ── OPERADORES ACTIVOS ────────────────────────────────────────
active = metrics[metrics['total_incoming'] >= 10].copy()

# ── UMBRALES Y FLAGS ──────────────────────────────────────────
p75_missed  = active['missed_rate_combined'].quantile(0.75)
p75_wait    = active['avg_wait_time'].quantile(0.75)
p25_out     = active[active['has_outgoing_role']]['outgoing_calls'].quantile(0.25)

active['flag_high_missed']  = active['missed_rate_combined'] > p75_missed
active['flag_high_wait']    = active['avg_wait_time']        > p75_wait
active['flag_low_outgoing'] = active.apply(
    lambda r: r['outgoing_calls'] < p25_out if r['has_outgoing_role'] else False, axis=1
)

active['inefficiency_score'] = (
    active['flag_high_missed'].astype(int) +
    active['flag_high_wait'].astype(int)   +
    active['flag_low_outgoing'].astype(int)
)
active['is_inefficient'] = active['inefficiency_score'] >= 2

# ── RESULTADO CON DESGLOSE ────────────────────────────────────
n_ineff = active['is_inefficient'].sum()
n_total = len(active)

print("=" * 60)
print("RESULTADO CON internal CORRECTAMENTE SEPARADA")
print("=" * 60)
print(f"Operadores activos    : {n_total}")
print(f"Operadores ineficaces : {n_ineff} ({n_ineff/n_total*100:.1f}%)")

print("\nComparación de tasas promedio por grupo:")
eff   = active[~active['is_inefficient']]
ineff = active[ active['is_inefficient']]

resumen = pd.DataFrame({
    'Eficaces':   [eff['missed_rate_external'].mean(),
                   eff['missed_rate_internal'].mean(),
                   eff['avg_wait_time'].mean()],
    'Ineficaces': [ineff['missed_rate_external'].mean(),
                   ineff['missed_rate_internal'].mean(),
                   ineff['avg_wait_time'].mean()]
}, index=['Tasa perdidas externas','Tasa perdidas internas','Tiempo espera (seg)'])

print(resumen.round(4).to_string())

## Paso 5: Prueba de las hipótesis estadísticas

In [ ]:
# ─── GRUPOS PARA LAS PRUEBAS ──────────────────────────────────
eficaces   = active[~active['is_inefficient']]
ineficaces = active[ active['is_inefficient']]

print("=" * 60)
print("GRUPOS DE ANÁLISIS")
print("=" * 60)
print(f"Operadores eficaces   : {len(eficaces):,}")
print(f"Operadores ineficaces : {len(ineficaces):,}")
print(f"Total activos         : {len(active):,}")

# ══════════════════════════════════════════════════════════════
# VERIFICACIÓN DE NORMALIDAD (decide qué prueba usar)
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("VERIFICACIÓN DE NORMALIDAD — Shapiro-Wilk")
print("(Si p < 0.05 → distribución NO normal → usar Mann-Whitney)")
print("=" * 60)

variables = {
    'Tasa llamadas perdidas' : 'missed_rate_combined',
    'Tiempo de espera'       : 'avg_wait_time',
    'Llamadas salientes'     : 'outgoing_calls',
}

for nombre, col in variables.items():
    # Shapiro-Wilk tiene límite de 5000 muestras, se muestrea si es necesario
    muestra_ef  = eficaces[col].dropna()
    muestra_in  = ineficaces[col].dropna()

    if len(muestra_ef) > 5000:
        muestra_ef = muestra_ef.sample(5000, random_state=42)
    if len(muestra_in) > 5000:
        muestra_in = muestra_in.sample(5000, random_state=42)

    _, p_ef = stats.shapiro(muestra_ef)
    _, p_in = stats.shapiro(muestra_in)

    normal_ef = "Normal OK" if p_ef >= 0.05 else "NO normal X"
    normal_in = "Normal OK" if p_in >= 0.05 else "NO normal X"

    print(f"\n  {nombre}")
    print(f"    Eficaces   → p = {p_ef:.4f}  {normal_ef}")
    print(f"    Ineficaces → p = {p_in:.4f}  {normal_in}")

print("\n→ Se usará Mann-Whitney U en todas las hipótesis")
print("  (prueba no paramétrica, no asume normalidad)")




In [ ]:
# ══════════════════════════════════════════════════════════════
# H1: TASA DE LLAMADAS PERDIDAS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("H1 — TASA DE LLAMADAS PERDIDAS (externas + internas)")
print("=" * 60)
print("H₀: No hay diferencia en la tasa de perdidas entre grupos")
print("H₁: Los ineficaces tienen MAYOR tasa de llamadas perdidas")
print(f"    (prueba unilateral — alternative='greater')\n")

h1_inef = ineficaces['missed_rate_combined'].dropna()
h1_efic = eficaces['missed_rate_combined'].dropna()

stat_h1, p_h1 = stats.mannwhitneyu(h1_inef, h1_efic, alternative='greater')

print(f"  Media eficaces          : {h1_efic.mean():.4f}  ({h1_efic.mean()*100:.2f}%)")
print(f"  Media ineficaces        : {h1_inef.mean():.4f}  ({h1_inef.mean()*100:.2f}%)")
print(f"  Mediana eficaces        : {h1_efic.median():.4f}")
print(f"  Mediana ineficaces      : {h1_inef.median():.4f}")
print(f"  Estadístico U           : {stat_h1:,.2f}")
print(f"  p-valor                 : {p_h1:.2e}")
print(f"  Nivel significancia (α) : 0.05")
print(f"  ¿Se rechaza H₀?         : {'SÍ — diferencia significativa' if p_h1 < 0.05 else 'NO — no hay evidencia suficiente'}")

# ── Tamaño del efecto: rank-biserial correlation ──────────────
n1, n2   = len(h1_inef), len(h1_efic)
r_h1     = 1 - (2 * stat_h1) / (n1 * n2)
magnitud = 'pequeño' if abs(r_h1) < 0.3 else 'mediano' if abs(r_h1) < 0.5 else 'grande'
print(f"  Tamaño del efecto (r)   : {r_h1:.4f}  → efecto {magnitud}")



In [ ]:
# ══════════════════════════════════════════════════════════════
# H2: TIEMPO DE ESPERA PROMEDIO
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("H2 — TIEMPO DE ESPERA PROMEDIO (solo llamadas externas)")
print("=" * 60)
print("H₀: No hay diferencia en el tiempo de espera entre grupos")
print("H₁: Los ineficaces tienen MAYOR tiempo de espera")
print(f"    (prueba unilateral — alternative='greater')\n")

h2_inef = ineficaces['avg_wait_time'].dropna()
h2_efic = eficaces['avg_wait_time'].dropna()

stat_h2, p_h2 = stats.mannwhitneyu(h2_inef, h2_efic, alternative='greater')

print(f"  Media eficaces          : {h2_efic.mean():.1f} seg")
print(f"  Media ineficaces        : {h2_inef.mean():.1f} seg")
print(f"  Mediana eficaces        : {h2_efic.median():.1f} seg")
print(f"  Mediana ineficaces      : {h2_inef.median():.1f} seg")
print(f"  Estadístico U           : {stat_h2:,.2f}")
print(f"  p-valor                 : {p_h2:.2e}")
print(f"  Nivel significancia (α) : 0.05")
print(f"  ¿Se rechaza H₀?         : {'SÍ — diferencia significativa' if p_h2 < 0.05 else 'NO — no hay evidencia suficiente'}")

n1, n2   = len(h2_inef), len(h2_efic)
r_h2     = 1 - (2 * stat_h2) / (n1 * n2)
magnitud = 'pequeño' if abs(r_h2) < 0.3 else 'mediano' if abs(r_h2) < 0.5 else 'grande'
print(f"  Tamaño del efecto (r)   : {r_h2:.4f}  → efecto {magnitud}")



In [ ]:
# ══════════════════════════════════════════════════════════════
# H3: LLAMADAS SALIENTES EXTERNAS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("H3 — LLAMADAS SALIENTES EXTERNAS (solo operadores con rol saliente)")
print("=" * 60)
print("H₀: No hay diferencia en llamadas salientes entre grupos")
print("H₁: Los eficaces realizan MÁS llamadas salientes")
print(f"    (prueba unilateral — alternative='greater')\n")

# Solo operadores con rol saliente
efic_out  = eficaces[eficaces['has_outgoing_role']]['outgoing_calls'].dropna()
inef_out  = ineficaces[ineficaces['has_outgoing_role']]['outgoing_calls'].dropna()

print(f"  Operadores eficaces con rol saliente   : {len(efic_out)}")
print(f"  Operadores ineficaces con rol saliente : {len(inef_out)}\n")

if len(inef_out) >= 5 and len(efic_out) >= 5:
    stat_h3, p_h3 = stats.mannwhitneyu(efic_out, inef_out, alternative='greater')

    print(f"  Media eficaces          : {efic_out.mean():,.0f} llamadas")
    print(f"  Media ineficaces        : {inef_out.mean():,.0f} llamadas")
    print(f"  Mediana eficaces        : {efic_out.median():,.0f} llamadas")
    print(f"  Mediana ineficaces      : {inef_out.median():,.0f} llamadas")
    print(f"  Estadístico U           : {stat_h3:,.2f}")
    print(f"  p-valor                 : {p_h3:.2e}")
    print(f"  Nivel significancia (α) : 0.05")
    print(f"  ¿Se rechaza H₀?         : {'SÍ — diferencia significativa' if p_h3 < 0.05 else 'NO — no hay evidencia suficiente'}")

    n1, n2   = len(efic_out), len(inef_out)
    r_h3     = 1 - (2 * stat_h3) / (n1 * n2)
    magnitud = 'pequeño' if abs(r_h3) < 0.3 else 'mediano' if abs(r_h3) < 0.5 else 'grande'
    print(f"  Tamaño del efecto (r)   : {r_h3:.4f}  → efecto {magnitud}")
else:
    print("  Advertencia:  Muestra insuficiente para la prueba")
    stat_h3, p_h3, r_h3 = None, None, None



In [ ]:
# ══════════════════════════════════════════════════════════════
# RESUMEN FINAL DE LAS 3 HIPÓTESIS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RESUMEN FINAL — PRUEBAS DE HIPÓTESIS")
print("=" * 60)
print(f"{'Hipótesis':<12} {'p-valor':>12} {'Sig (α=0.05)':>14} {'Efecto (r)':>12} {'Magnitud':>10}")
print("-" * 60)

def magnitud_efecto(r):
    if r is None: return 'N/A'
    return 'pequeño' if abs(r) < 0.3 else 'mediano' if abs(r) < 0.5 else 'grande'

hipotesis = [
    ('H1 (perdidas)', p_h1, r_h1),
    ('H2 (espera)',   p_h2, r_h2),
    ('H3 (salientes)',p_h3, r_h3),
]

for nombre, p, r in hipotesis:
    if p is not None:
        sig    = 'Rechaza H₀' if p < 0.05 else 'No rechaza'
        r_str  = f"{r:.4f}"
        p_str  = f"{p:.2e}"
    else:
        sig, r_str, p_str = 'N/A', 'N/A', 'N/A'
    print(f"{nombre:<12} {p_str:>12} {sig:>14} {r_str:>12} {magnitud_efecto(r):>10}")

## Paso 6: Análisis de clientes afectados por operadores ineficaces

In [ ]:

# ══════════════════════════════════════════════════════════════
# PASO 1: IDENTIICAR OPERADORES INEFICACES
# ══════════════════════════════════════════════════════════════
operadores_ineficaces = set(active[active['is_inefficient']].index)
operadores_eficaces   = set(active[~active['is_inefficient']].index)

print("\n" + "=" * 60)
print("PASO 1 — IDENTIICAR OPERADORES INEFICACES")
print("=" * 60)

print(f"\nOperadores ineficaces identificados: {len(operadores_ineficaces)}")


# ══════════════════════════════════════════════════════════════
# PASO 2: VINCULAR OPERADORES INEFICACES CON CLIENTES
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 2 — VINCULAR OPERADORES INEFICACES CON CLIENTES")
print("=" * 60)

# Marcar cada registro con si su operador es ineficaz
calls_clean['op_is_inefficient'] = calls_clean['operator_id'].isin(operadores_ineficaces)
calls_clean['op_is_efficient']   = calls_clean['operator_id'].isin(operadores_eficaces)

# Clientes expuestos a al menos un operador ineficaz
clientes_expuestos = (
    calls_clean[calls_clean['op_is_inefficient']]
    ['user_id'].unique()
)

print(f"Clientes únicos en el dataset        : {calls_clean['user_id'].nunique():,}")
print(f"Clientes expuestos a ineficaces      : {len(clientes_expuestos):,}")
print(f"Porcentaje afectados                 : {len(clientes_expuestos)/calls_clean['user_id'].nunique()*100:.1f}%")


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 3: MÉTRICAS DE IMPACTO POR CLIENTE
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 3 — MÉTRICAS DE IMPACTO POR CLIENTE")
print("=" * 60)

# Llamadas perdidas externas entrantes que sufrió cada cliente
# por operadores ineficaces vs eficaces
def metricas_cliente(df_filtrado):
    """Calcula métricas de calidad de servicio por cliente."""

    # Llamadas perdidas externas entrantes
    missed = (
        df_filtrado[(df_filtrado['direction'] == 'in') &
                    (df_filtrado['is_missed_call'] == True) &
                    (df_filtrado['internal'] == False)]
        .groupby('user_id')['calls_count'].sum()
        .rename('llamadas_perdidas')
    )
    # Total llamadas entrantes externas
    total_in = (
        df_filtrado[(df_filtrado['direction'] == 'in') &
                    (df_filtrado['internal'] == False)]
        .groupby('user_id')['calls_count'].sum()
        .rename('total_entrantes')
    )
    # Tiempo de espera promedio
    wait = (
        df_filtrado[(df_filtrado['direction'] == 'in') &
                    (df_filtrado['is_missed_call'] == False) &
                    (df_filtrado['internal'] == False)]
        .groupby('user_id')['wait_time'].mean()
        .rename('espera_promedio')
    )
    # Operadores distintos que atendieron al cliente
    ops = (
        df_filtrado.groupby('user_id')['operator_id']
        .nunique().rename('operadores_distintos')
    )

    result = (
        pd.DataFrame(index=df_filtrado['user_id'].unique())
        .join(missed).join(total_in).join(wait).join(ops)
        .fillna(0)
    )
    result.index.name = 'user_id'
    result['tasa_perdidas'] = np.where(
        result['total_entrantes'] > 0,
        result['llamadas_perdidas'] / result['total_entrantes'], 0
    )
    return result

# Métricas cuando el cliente fue atendido por ineficaces
impacto_inef = metricas_cliente(calls_clean[calls_clean['op_is_inefficient']])
impacto_inef.columns = [f"{c}_inef" for c in impacto_inef.columns]

# Métricas cuando el cliente fue atendido por eficaces
impacto_efic = metricas_cliente(calls_clean[calls_clean['op_is_efficient']])
impacto_efic.columns = [f"{c}_efic" for c in impacto_efic.columns]

# Combinar y agregar info de clientes
impacto = impacto_inef.join(impacto_efic, how='outer').fillna(0)
impacto = impacto.join(
    clients.set_index('user_id')[['tariff_plan','date_start']], how='left'
)
impacto['fue_expuesto'] = impacto.index.isin(clientes_expuestos)

print(f"\nClientes con métricas calculadas: {len(impacto):,}")
print(f"\nPrimeras 5 filas del DataFrame de impacto:")
print(impacto[['llamadas_perdidas_inef','espera_promedio_inef',
               'llamadas_perdidas_efic','espera_promedio_efic',
               'tariff_plan','fue_expuesto']].head())


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 4: ANÁLISIS DE IMPACTO POR PLAN TARIFARIO
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 4 — IMPACTO POR PLAN TARIFARIO")
print("=" * 60)

expuestos = impacto[impacto['fue_expuesto']].copy()

impacto_tarifa = (
    expuestos.groupby('tariff_plan')
    .agg(
        clientes_afectados    = ('llamadas_perdidas_inef', 'count'),
        total_llamadas_perdidas = ('llamadas_perdidas_inef', 'sum'),
        promedio_perdidas_cliente = ('llamadas_perdidas_inef', 'mean'),
        espera_promedio_seg   = ('espera_promedio_inef', 'mean'),
        tasa_perdidas_media   = ('tasa_perdidas_inef', 'mean'),
    )
    .round(2)
    .sort_values('total_llamadas_perdidas', ascending=False)
)

print(impacto_tarifa.to_string())

# Totales por plan (del total de clientes registrados)
total_por_plan = clients.groupby('tariff_plan')['user_id'].count().rename('total_clientes')
impacto_tarifa = impacto_tarifa.join(total_por_plan)
impacto_tarifa['pct_afectados'] = (
    impacto_tarifa['clientes_afectados'] / impacto_tarifa['total_clientes'] * 100
).round(1)

print("\nPorcentaje de clientes afectados por plan:")
print(impacto_tarifa[['total_clientes','clientes_afectados','pct_afectados',
                        'total_llamadas_perdidas','espera_promedio_seg']].to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 5: TOP 10 CLIENTES MÁS AFECTADOS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 5 — TOP 10 CLIENTES MÁS AFECTADOS")
print("=" * 60)

top10_clientes = (
    expuestos[['llamadas_perdidas_inef','espera_promedio_inef',
               'tasa_perdidas_inef','operadores_distintos_inef','tariff_plan']]
    .sort_values('llamadas_perdidas_inef', ascending=False)
    .head(10)
)
top10_clientes.columns = ['Llamadas perdidas','Espera prom (seg)',
                           'Tasa perdidas','Operadores inef distintos','Plan']
print(top10_clientes.round(2).to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 6: ANTIGÜEDAD DEL CLIENTE vs EXPOSICIÓN A INEFICACES
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 6 — ANTIGÜEDAD DEL CLIENTE vs IMPACTO")
print("=" * 60)

fecha_ref = pd.to_datetime(calls_clean['date'], utc=True).max()

impacto['date_start'] = pd.to_datetime(impacto['date_start'], utc=True)

impacto['antiguedad_dias'] = (fecha_ref - impacto['date_start']).dt.days
impacto['segmento_antiguedad'] = pd.cut(
    impacto['antiguedad_dias'],
    bins=[0, 180, 365, 730, 9999],
    labels=['< 6 meses', '6–12 meses', '1–2 años', '> 2 años']
)

antiguedad_impacto = (
    impacto[impacto['fue_expuesto']]
    .groupby('segmento_antiguedad', observed=True)
    .agg(
        clientes          = ('llamadas_perdidas_inef', 'count'),
        perdidas_promedio = ('llamadas_perdidas_inef', 'mean'),
        espera_promedio   = ('espera_promedio_inef', 'mean'),
    )
    .round(2)
)
print(antiguedad_impacto.to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASO 7: PRUEBA ESTADÍSTICA — ¿EL PLAN TARIFARIO INFLUYE
#         EN LA CANTIDAD DE LLAMADAS PERDIDAS POR INEFICACES?
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PASO 7 — PRUEBA ESTADÍSTICA: PLAN TARIFARIO vs IMPACTO")
print("=" * 60)
print("H₀: No hay diferencia en llamadas perdidas entre planes")
print("H₁: Al menos un plan tarifario sufre más llamadas perdidas")
print("Prueba: Kruskal-Wallis (no paramétrica, más de 2 grupos)\n")

grupos_plan = [
    expuestos[expuestos['tariff_plan'] == plan]['llamadas_perdidas_inef'].dropna()
    for plan in expuestos['tariff_plan'].dropna().unique()
]
grupos_validos = [g for g in grupos_plan if len(g) >= 5]

if len(grupos_validos) >= 2:
    stat_kw, p_kw = stats.kruskal(*grupos_validos)
    print(f"  Estadístico H  : {stat_kw:.4f}")
    print(f"  p-valor        : {p_kw:.4f}")
    print(f"  ¿Rechaza H₀?   : {'SÍ — hay diferencias entre planes' if p_kw < 0.05 else 'NO — no hay diferencias significativas'}")
else:
    print("  Advertencia: Grupos insuficientes para la prueba")


In [ ]:
# ══════════════════════════════════════════════════════════════
# RESUMEN EJECUTIVO
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RESUMEN EJECUTIVO — IMPACTO EN CLIENTES")
print("=" * 60)
print(f"  Clientes totales registrados          : {len(clients):,}")
print(f"  Clientes expuestos a ineficaces        : {len(clientes_expuestos):,}  ({len(clientes_expuestos)/len(clients)*100:.1f}%)")
print(f"  Total llamadas perdidas por ineficaces : {int(expuestos['llamadas_perdidas_inef'].sum()):,}")
print(f"  Tiempo de espera promedio (ineficaces) : {expuestos['espera_promedio_inef'].mean():.1f} seg")
print(f"  Tiempo de espera promedio (eficaces)   : {impacto['espera_promedio_efic'].mean():.1f} seg")
print(f"  Plan más afectado                      : {impacto_tarifa['total_llamadas_perdidas'].idxmax()}")

## Paso 7: Visualización de resultados

In [ ]:
# ══════════════════════════════════════════════════════════════
# PALETA DE COLORES CORPORATIVA
# ══════════════════════════════════════════════════════════════
COLOR_EFICAZ     = '#2DC653'   # verde
COLOR_INEFICAZ   = '#E63946'   # rojo
COLOR_ALERTA     = '#F4C430'   # amarillo
COLOR_CRITICO    = '#B5000A'   # rojo oscuro
COLOR_UMBRAL     = '#0077A8'   # azul
COLOR_FONDO      = '#F8FAFC'
COLOR_TITULO     = '#0D1B2A'
COLOR_SUBTITULO  = '#3A5F72'

PLAN_COLORES = {'A': '#00B4D8', 'B': '#0077A8', 'C': '#004E89'}


In [ ]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 1 — DISPERSIÓN: OPERADORES
# Por qué dispersión:
#   Permite visualizar SIMULTÁNEAMENTE dos métricas continuas
#   (tasa de perdidas vs tiempo de espera) para cada operador,
#   revelar clusters, outliers y la separación entre grupos.
#   Un gráfico de barras no podría mostrar la relación entre
#   ambas variables al mismo tiempo.
# ══════════════════════════════════════════════════════════════
# 1. Calculamos el umbral en volumen absoluto y tiempo de espera
p75_missed_count = active["missed_total"].quantile(0.75)
val_p75_wait = p75_wait

# 2. Configurar el gráfico con el fondo corporativo tenue
fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
fig.patch.set_facecolor("#f4f6f9")  # Fondo de la imagen completa
ax.set_facecolor("#f4f6f9")  # Fondo del área de trazado

# Separar datos
eficaces = active[~active["is_inefficient"]]
ineficaces = active[active["is_inefficient"]]

# 3. Graficar los operadores (Gris tenue vs Rojo corporativo)
ax.scatter(
    eficaces["missed_total"],
    eficaces["avg_wait_time"],
    c="#b0bec5",  # Gris azulado tenue corporativo
    s=80,
    alpha=0.7,
    edgecolors="#78909c",
    linewidths=0.5,
    zorder=2,
    label="Operadores Eficaces",
)

ax.scatter(
    ineficaces["missed_total"],
    ineficaces["avg_wait_time"],
    c="#d32f2f",  # Rojo brillante y profesional
    s=105,
    alpha=0.95,
    edgecolors="#5d0000",
    linewidths=1,
    zorder=2,
    label="Operadores Ineficaces",
)

# 4. Límites de los ejes ajustados al peor operador real con margen
ax.set_xlim(-0.5, active["missed_total"].max() * 1.1)
ax.set_ylim(-2, active["avg_wait_time"].max() * 1.1)

# 5. CONFIGURACIÓN DE LOS DOS UMBRALES SOLICITADOS (AMBOS ROJOS)
# Umbral de Pérdidas (Vertical): LÍNEA DISCONTINUA (RAYAS)
ax.axvline(
    x=p75_missed_count,
    color="#d32f2f",
    linestyle="--",  # Discontinua
    linewidth=2.5,
    zorder=3,
)

# Umbral de Espera (Horizontal): LÍNEA PUNTEADA (PUNTOS)
ax.axhline(
    y=val_p75_wait,
    color="#d32f2f",
    linestyle=":",  # Punteada
    linewidth=2.5,
    zorder=3,
)

# 6. REJILLA CORPORATIVA Y FORMATO DE EJES
ax.xaxis.set_major_formatter("{x:.0f}")
ax.yaxis.set_major_formatter("{x:.0f}s")
# Rejilla blanca sutil sobre el fondo gris azulado
ax.grid(True, linestyle="--", alpha=0.6, color="#ffffff", zorder=1)

# 7. TÍTULO CON TIPOGRAFÍA ELEGANTE Y TEXTOS GENERALES
ax.set_title(
    "Distribución de Operadores: Volumen de Pérdidas vs Tiempo de Espera",
    fontname="Georgia",
    fontsize=16,
    fontweight="bold",
    color="#263238",
    pad=20,
)

# Etiquetas de ejes limpias
ax.set_xlabel(
    "Cantidad Total de Llamadas Perdidas (Unidades)",
    fontsize=11,
    color="#37474f",
    labelpad=10,
)
ax.set_ylabel(
    "Tiempo de Espera Promedio (Segundos)",
    fontsize=11,
    color="#37474f",
    labelpad=10,
)

# Ajustar color de los números de los ejes
ax.tick_params(colors="#37474f", labelsize=10)

# 8. LEYENDA ACTUALIZADA CON LOS NUEVOS ESTILOS DE LÍNEA
handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#b0bec5",
        markeredgecolor="#78909c",
        markersize=8,
        label="Operadores Eficaces",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#d32f2f",
        markeredgecolor="#5d0000",
        markersize=9,
        label="Operadores Ineficaces",
    ),
    Line2D(
        [0],
        [0],
        color="#d32f2f",
        linestyle="--",  # Refleja la línea discontinua en la leyenda
        linewidth=2,
        label=f"Umbral Pérdidas ({p75_missed_count:.0f} llamadas)",
    ),
    Line2D(
        [0],
        [0],
        color="#d32f2f",
        linestyle=":",  # Refleja la línea punteada en la leyenda
        linewidth=2.5,
        label=f"Umbral Espera ({val_p75_wait:.1f}s)",
    ),
]

# Colocamos la leyenda con fondo blanco para que resalte del gris corporativo
ax.legend(
    handles=handles,
    loc="upper right",
    frameon=True,
    facecolor="#ffffff",
    edgecolor="none",
    fontsize=10,
)

# 9. GUARDAR EL GRÁFICO
plt.savefig(
    "reporte_operadores_ineficaces.png",
    dpi=300,
    facecolor=fig.get_facecolor(),
    edgecolor="none",
)

plt.show()
print("Gráfico 1 guardado: reporte_operadores_ineficaces.png")

In [ ]:
# ══════════════════════════════════════════════════════════════
# GRÁFICO 2 — BARRAS HORIZONTALES: TOP 10 CLIENTES AFECTADOS
# Por qué barras horizontales:
#   Los IDs de clientes son etiquetas largas que en barras
#   verticales se solaparían. Las barras horizontales permiten
#   leer con claridad cada etiqueta, comparar magnitudes de
#   izquierda a derecha y ordenar de mayor a menor impacto
#   de forma natural (de arriba hacia abajo).
#   Un gráfico de líneas no tendría sentido porque los clientes
#   no tienen orden secuencial entre sí.
# ══════════════════════════════════════════════════════════════
top10_clientes = (
    expuestos[['llamadas_perdidas_inef', 'espera_promedio_inef',
               'tasa_perdidas_inef', 'tariff_plan']]
    .sort_values('llamadas_perdidas_inef', ascending=False)
    .head(10)
    .copy()
)
top10_clientes = top10_clientes.reset_index()
top10_clientes['etiqueta'] = (
    'Cliente ' + top10_clientes['user_id'].astype(str) +
    '  [Plan ' + top10_clientes['tariff_plan'].fillna('N/A') + ']'
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(COLOR_FONDO)
fig.suptitle(
    'Top 10 Clientes más Afectados por Operadores Ineficaces',
    fontsize=15, fontweight='bold', color=COLOR_TITULO, y=1.01
)

# ── Panel izquierdo: Llamadas perdidas ────────────────────────
ax1 = axes[0]
ax1.set_facecolor(COLOR_FONDO)

colores_barras = [
    PLAN_COLORES.get(plan, '#8EABB5')
    for plan in top10_clientes['tariff_plan'].fillna('N/A')
]

bars1 = ax1.barh(
    top10_clientes['etiqueta'],
    top10_clientes['llamadas_perdidas_inef'],
    color=colores_barras,
    edgecolor='white', linewidth=0.6,
    height=0.65
)

# Valor al final de cada barra
for bar, val in zip(bars1, top10_clientes['llamadas_perdidas_inef']):
    ax1.text(
        bar.get_width() + top10_clientes['llamadas_perdidas_inef'].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{int(val):,}',
        va='center', ha='left',
        fontsize=9, color=COLOR_TITULO, fontweight='bold'
    )

ax1.invert_yaxis()
ax1.set_xlabel('Llamadas Perdidas (cantidad)', fontsize=11,
               color=COLOR_SUBTITULO, labelpad=8)
ax1.set_title('Por Llamadas Perdidas', fontsize=12,
              color=COLOR_TITULO, fontweight='bold', pad=10)
ax1.tick_params(axis='y', labelsize=9, colors=COLOR_TITULO)
ax1.tick_params(axis='x', labelsize=9, colors=COLOR_SUBTITULO)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#D0D0D0')
ax1.grid(axis='x', linestyle='--', linewidth=0.5,
         color='#D0D0D0', alpha=0.7)
ax1.set_xlim(0, top10_clientes['llamadas_perdidas_inef'].max() * 1.18)

# ── Panel derecho: Tiempo de espera promedio ──────────────────
ax2 = axes[1]
ax2.set_facecolor(COLOR_FONDO)

# Reordenar por tiempo de espera para el segundo panel
top10_espera = (
    expuestos[['espera_promedio_inef', 'tasa_perdidas_inef', 'tariff_plan']]
    .sort_values('espera_promedio_inef', ascending=False)
    .head(10)
    .reset_index()
)
top10_espera['etiqueta'] = (
    'Cliente ' + top10_espera['user_id'].astype(str) +
    '  [Plan ' + top10_espera['tariff_plan'].fillna('N/A') + ']'
)

colores_espera = [
    PLAN_COLORES.get(plan, '#8EABB5')
    for plan in top10_espera['tariff_plan'].fillna('N/A')
]

bars2 = ax2.barh(
    top10_espera['etiqueta'],
    top10_espera['espera_promedio_inef'],
    color=colores_espera,
    edgecolor='white', linewidth=0.6,
    height=0.65
)

for bar, val in zip(bars2, top10_espera['espera_promedio_inef']):
    ax2.text(
        bar.get_width() + top10_espera['espera_promedio_inef'].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val:,.1f} seg',
        va='center', ha='left',
        fontsize=9, color=COLOR_TITULO, fontweight='bold'
    )

ax2.invert_yaxis()
ax2.set_xlabel('Tiempo de Espera Promedio (segundos)', fontsize=11,
               color=COLOR_SUBTITULO, labelpad=8)
ax2.set_title('Por Tiempo de Espera Promedio', fontsize=12,
              color=COLOR_TITULO, fontweight='bold', pad=10)
ax2.tick_params(axis='y', labelsize=9, colors=COLOR_TITULO)
ax2.tick_params(axis='x', labelsize=9, colors=COLOR_SUBTITULO)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#D0D0D0')
ax2.grid(axis='x', linestyle='--', linewidth=0.5,
         color='#D0D0D0', alpha=0.7)
ax2.set_xlim(0, top10_espera['espera_promedio_inef'].max() * 1.22)

# Leyenda de planes (compartida)
leyenda_planes = [
    mpatches.Patch(color=color, label=f'Plan {plan}')
    for plan, color in PLAN_COLORES.items()
]
fig.legend(
    handles=leyenda_planes,
    loc='lower center', ncol=3,
    fontsize=10, framealpha=0.9,
    edgecolor='#D0D0D0', facecolor='white',
    bbox_to_anchor=(0.5, -0.04)
)

plt.tight_layout()
plt.savefig('barras_clientes_afectados.png', dpi=150,
            bbox_inches='tight', facecolor=COLOR_FONDO)
plt.show()
print("Gráfico 2 guardado: barras_clientes_afectados.png")

## Paso 8: Conclusiones

In [ ]:
# 1. Calculamos los valores numéricos exactos de tus datos
valor_llamadas_p75 = int(active["missed_total"].quantile(0.75))
valor_espera_p75 = round(p75_wait, 1)

# 2. Imprimimos el texto completo con los datos integrados automáticamente
print("=" * 80)
print("             RESUMEN EJECUTIVO: EVALUACIÓN DE EFICIENCIA OPERATIVA")
print("=" * 80)
print(f"""
1. REGLA DE EVALUACIÓN
Para ser evaluado, un operador debe ser activo (mínimo 10 llamadas entrantes). 
El sistema activa las alertas si el operador cruza los siguientes límites:

* Alerta de Pérdidas: El operador acumula MÁS DE {valor_llamadas_p75} llamadas perdidas totales.
  (Este es el límite del Percentil 75; el 25% de los operadores con peor desempeño 
  pierde una cantidad igual o mayor a esta cifra).

* Alerta de Espera: Su tiempo de espera promedio supera los {valor_espera_p75} segundos.

* Alerta Saliente: Realiza menos de 15 llamadas salientes (Percentil 25).

Criterio de Ineficacia: Un operador es declarado Ineficaz si acumula 2 o más alertas.

--------------------------------------------------------------------------------

2. INDICADORES CLAVE DE IMPACTO (KPIS)
* Base de Clientes Afectada: De un total de 732 clientes registrados, 37 cuentas (5.1%) 
  estuvieron expuestas directamente a la atención de operadores ineficaces.
* Fuga de Contacto: Se registraron 430 llamadas perdidas adjudicadas exclusivamente 
  al grupo de operadores ineficaces.
* Penalización en Tiempo de Espera: 
  - Operadores Ineficaces: promedio de 172.5 segundos (superando drásticamente el límite de {valor_espera_p75}s).
  - Operadores Eficaces: promedio de 68.5 segundos (dentro del estándar seguro).
  - Brecha de Rendimiento: El grupo ineficaz hace esperar al cliente un 151.8% más de tiempo.
* Segmento Crítico: El Plan A es la categoría de producto/servicio más afectada.

--------------------------------------------------------------------------------

3. CONCLUSIONES ESTRATÉGICAS

* Transparencia en el Castigo Operativo: La gráfica de dispersión confirma que la 
  penalización es justa. No se señala a nadie por un error aislado; para activar la 
  alerta de fugas, un operador debe superar la barrera de {valor_llamadas_p75} llamadas perdidas. 
  El grupo ineficaz absorbió un total masivo de 430 llamadas perdidas, evidenciando 
  un problema de rendimiento sistémico y no un evento fortuito.

* Riesgo de Churn Focalizado: Perder 430 interacciones telefónicas representa cientos 
  de oportunidades de venta o soporte técnico frustradas. Al estar concentradas en 
  los clientes del Plan A, la empresa enfrenta un riesgo inminente de cancelación 
  de cuentas (churn) en su segmento más sensible.

* Inacción Comercial (Salientes): El perfil del operador ineficaz se consolida por su 
  falta de proactividad. Además de registrar tiempos de espera intolerables (172.5s) 
  y perder llamadas, no cumplen con el mínimo de 15 llamadas salientes, dejando 
  desatendidas las campañas de contacto activo.

* Brecha de Capacitación: El hecho de que el grupo eficaz logre un promedio sobresaliente 
  de 68.5 segundos demuestra que las metas son viables con las herramientas actuales. 
  La ineficiencia está ligada estrictamente a un rezago individual en la destreza 
  técnica o en la gestión de los tiempos de operación.
""")
print("=" * 80)


## Paso 9: Recomendaciones

Para mitigar el daño y corregir el rumbo de la empresa de telecomunicaciones, 
se sugiere proponer tres acciones inmediatas basado en los resultados obtenidos:
1. **Reenrutamiento Operativo y Restricción de Perfiles:** Modificar la matriz de ruteo del sistema telefónico 
   (IVR) para asegurar que las cuentas del Plan A sean atendidas únicamente por el 
   grupo de operadores eficaces. Se sugiere que ningún operador que se sitúe en la lista de ineficaces pueda tener asignadas 
   llamadas de los planes de tarifa premium (A incluso B). Estos empleados deben ser movidos temporalmente a un esquema de 
   llamadas de bajo impacto o campañas internas.
2. **Campaña de Retención Activa:** Extraer los IDs de los 37 clientes expuestos para ejecutar 
   un contacto proactivo de control de daños antes de que inicien trámites de baja.
3. **Programa Espacial de Capacitación y Auditoría Técnica:** Convocar a los operadores ineficaces identificados 
   para realizar una auditoría de llamadas grabadas y resolver cuellos de botella técnicos, para descartar fallas de hardware 
   (diademas o mala conexión) o reentrenar en gestión de colas.


**Mantenimiento y Control de Calidad de Datos (TI):**
Considerar institucionalizar la eliminación de duplicados en los reportes mensuales de TI. 
Haber detectado un 9.09% de registros espejo (4,900 filas) implica que los sistemas de logs sufren de redundancia, 
lo cual debe corregirse en el servidor para evitar reportes inflados en el futuro.

### Recomendaciones Finales
* **Monitoreo semanal automatizado**
Implementar un dashboard que calcule el score de ineficiencia semanalmente, con alertas automaticas para scores >= 2 y revision inmediata para scores = 3.
* **Plan de capacitacion por score**
Score 1 (32 operadores): seguimiento mensual. Score 2: capacitacion en 30 dias. Score 3: intervencion inmediata con supervision directa del operador.
* **Revision de asignacion para clientes Plan A**
El Plan A concentra la mayor cantidad de llamadas perdidas por ineficaces. Evaluar si los operadores asignados tienen la capacidad y capacitacion adecuadas.
* **Ampliar el analisis de los 37 clientes afectados**
Implementar analisis de churn risk, cohortes de registro y correlacion plan-calidad para los clientes expuestos, priorizando los del top 10 mas afectados.
* **Revision del criterio de llamadas salientes** 
Validar con el negocio si el umbral P25 es adecuado para el rol saliente, ya que algunos operadores pueden tener roles mixtos no reflejados en el dataset.


#  Fuentes (documentación, artículos, etc) utilizadas para el proyecto
### Documentación Técnica (Python y Ciencia de Datos)

1. **Pandas Official Documentation (pandas.DataFrame.drop_duplicates)**
* **Descripción** Documentación oficial utilizada para validar la certeza matemática del método .drop_duplicates(), asegurando que evalúa la coincidencia del 100% de las columnas por defecto para eliminar la redundancia de datos [google:python_interpreter].
* **Uso en el proyecto:** Fase de limpieza (reducción de 53,902 a 49,002 registros).

2. **Python Math Module Documentation (math.ceil)**
* **Descripción:** Guía de referencia de las funciones matemáticas nativas de Python. Se consultó para aplicar el redondeo estricto hacia el número entero superior, transformando límites continuos en variables discretas de negocio.
* **Uso en el proyecto:** Recalibración del límite de llamadas perdidas (> 4.0).

3. **SciPy Stats Documentation (scipy.stats.ttest_ind)**
* **Descripción:** Manual técnico para la implementación de pruebas estadísticas. Se utilizó específicamente para configurar la Prueba \(t\) de Student para muestras independientes con el ajuste de Welch (equal_var=False), adecuado para grupos con varianzas y tamaños diferentes.
* **Uso en el proyecto:** Fase de validación y pruebas de hipótesis.

4. **Tableau Knowledge Base: Creating Calculated Fields & Bins**
* **Descripción:** Guía oficial de usuario de Tableau para el desarrollo de inteligencia de negocios. Se utilizó para estructurar la fórmula de la métrica raíz ([Total Call Duration] - [Call Duration]) y la creación de contenedores numéricos para el histograma.
* **Uso en el proyecto:** Maquetación del Dashboard interactivo.
5. **Pandas Documentation — pandas.pydata.org**
pandas.pydata.org/docs
* **Descripción:** Como agrupar registros por operador con groupby() para calcular metricas agregadas calcular percentiles con quantile() para definir umbrales de ineficiencia. Como manejar valores nulos con fillna() y eliminar duplicados con drop_duplicates()

6. **SciPy Stats — docs.scipy.org/doc/scipy/reference/stats**
docs.scipy.org/doc/scipy
* **Descripcipon:** Como implementar la prueba Mann-Whitney U para comparar grupos no parametricos normalidad con Shapiro-Wilk antes de elegir la prueba correcta. Como interpretar el estadistico U y el p-valor en pruebas unilaterales | Como verificar

7. **NumPy Documentation — numpy.org/doc**
numpy.org/doc
* **Descripción:** Como usar np.where() para calcular tasas de perdidas evitando division por cero | Como escalar valores entre rangos para el tamano proporcional de puntos en scatter.

8. **Matplotlib Documentation — matplotlib.org/stable**
matplotlib.org/stable/api
* **Descripción:** Como recortar ejes al P95 para visualizar la masa principal de datos (Solucion B) vacia adicional en Jupyter con %matplotlib inline. Como configurar hexbin para visualizar densidad de puntos superpuestos | Como evitar la figuraFUENTES Y 

9. **Python for Data Analysis — Wes McKinney (O'Reilly, 3ra ed.)**
wesmckinney.com/book
* Descripcioón:** Metodologias de limpieza de datos: duplicados, nulos y tipos inconsistentes como internal cargada como object. Buenas practicas para EDA: describe(), value_counts() y deteccion de anomalias

10. **Discovering Statistics Using IBM SPSS — Andy Field (SAGE Publications)**
discoveringstatistics.com
* **Descripción:** Criterios para elegir entre pruebas parametricas y no parametricas segun normalidad | Como calcular e interpretar el tamano del efecto (rank-biserial r) en Mann-Whitney


11. **Seaborn / Matplotlib — Guia de Visualizacion Estadistica**
seaborn.pydata.org / matplotlib.org/stable/gallery
* **Descripción:** Cuando usar graficos de dispersion vs densidad segun el volumen de datos disponibles Solucion al amontonamiento visual: zoom al P95 para audiencias ejecutivas no tecnicas. Como usar barras horizontales para etiquetas largas en rankings de clientes afectados |CON

### Estándares de la Industria y Artículos de Negocio (Call Centers)
12. **Towards Data Science — Call Center KPI Best Practices**
towardsdatascience.com
* **Descripción:** Que metricas son estandar en la industria para medir eficiencia de operadores en contact centers fijos | Por que separar llamadas internas y externas para un analisis mas preciso y sin sesgo. Como definir umbrales de rendimiento basados en percentiles en lugar de valores
12. **International Customer Management Institute (ICMI) – Call Center Service Level Standards**
* **Descripción:** Artículo especializado en los estándares globales de calidad en centros de contacto. Detalla la métrica tradicional del acuerdo de nivel de servicio (SLA) y cómo el aumento del tiempo de espera impacta directamente en la experiencia del cliente (CX).
* **Uso en el proyecto:** Justificación de negocio para el umbral de tiempo de espera (> 152.1 segundos).

13. **Harvard Business Review – The Customer Service Life Cycle and Churn Management**
* **Descripción:** Artículo de investigación sobre la retención de clientes. Analiza el impacto financiero que sufren las empresas de servicios cuando las cuentas VIP experimentan degradación en la atención, gatillando el abandono de la marca (Churn).
* **Uso en el proyecto:** Justificación para el análisis de vulnerabilidad comercial (Top 10 de clientes afectados).

14. **MetricNet – Call Center Key Performance Indicators (KPIs) Benchmarking**
* **Descripción:** Informe global de evaluación comparativa que define los rangos normales de llamadas perdidas (Abandonment Rate) y productividad de agentes de emisión (Outbound Agent Productivity).
* **Uso en el proyecto:** Base conceptual para definir los perfiles Inbound y Outbound por separado.

15. **NIST (National Institute of Standards and Technology) – Engineering Statistics Handbook: The Normal Distribution**
* **Descripción:** Manual académico de estadística aplicada. Se utilizó como base metodológica para justificar por qué sumar o restar una desviación estándar a la media (Media +/- 1 Std) es un criterio robusto para aislar anomalías de rendimiento en una población.
* **Uso en el proyecto:** Modelado estadístico de los límites dinámicos.

### Paso 10. Opcional: Preparación para Tableu

In [ ]:
# guardo el resultado limpio en un archivo nuevo de 49,002 filas netas
calls_clean.to_csv('telecom_dataset_clean.csv', index=False)